# Circle radial-resolution measurement

Use this notebook for NCI images containing many circular targets. Click one point inside each circle. The notebook refines each clicked seed to a circle center, extracts an azimuthally averaged radial edge-spread function (ESF), fits the ESF with a Gaussian-blurred step edge, and reports the inferred line-spread sigma, FWHM, 10-90 edge width, MTF10 frequency, and an MTF10 half-period resolution estimate.

The click only needs to be inside the circle. Avoid circles cut off by the image edge or strongly overlapping with another circle unless you want to inspect them manually afterward.

## 1. Settings

Start by changing only `IMAGE_PATH`, `PIXEL_SIZE_UM`, and possibly `FEATURE_POLARITY`. The current default points to the summed final exp303 CirclesNew image found on this machine.

In [ ]:
from pathlib import Path
import os

IMAGE_PATH = Path('/data01/data_acquisition/data/exp303_ZnSAg_15_5_CirclesNew_2.8Lens/exp303_ZnSAg_15_5_CirclesNew_2.8Lens_00000/final/exp303_ZnSAg_15_5_CirclesNew_2.8Lens_00000_sumImage_5to1000.tiff')

# Set to the detector/image pixel size in um if you want physical units.
# Leave as None to report only pixel units.
PIXEL_SIZE_UM = None

# Use 'bright' for bright circles on dark background, 'dark' for dark circles on bright background.
FEATURE_POLARITY = 'bright'

# Display only: downsample the large image in the interactive picker for responsiveness.
# Coordinates are still stored in full-resolution image pixels.
DISPLAY_DOWNSAMPLE = 2
DISPLAY_PERCENTILES = (0.5, 99.7)

# Center-finding parameters. Increase SEARCH_RADIUS_PX if the largest circles are not fully inside the local crop.
SEARCH_RADIUS_PX = 260
SMOOTH_SIGMA_PX = 1.2
MIN_COMPONENT_AREA_PX = 80
SEED_SNAP_RADIUS_PX = 25

# Radial-profile and fit parameters.
N_ANGLES = 720
RADIAL_STEP_PX = 0.25
EDGE_WINDOW_IN_PX = 25       # ESF fit range inside the detected edge
EDGE_WINDOW_OUT_PX = 30      # ESF fit range outside the detected edge; keep modest to avoid neighboring circles
EDGE_POINT_SEARCH_PX = 18    # center refinement searches for the edge near the initial radius
MAX_CENTER_REFINE_ITER = 3

OUTPUT_DIR = Path('/data01/software/toolbox/image analysis/circle_resolution_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep Matplotlib cache/config files inside the writable workspace on this machine.
MPLCONFIGDIR = OUTPUT_DIR / 'matplotlib_cache'
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

IMAGE_PATH, OUTPUT_DIR

## 2. Imports and helper functions

In [ ]:

import math
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import curve_fit
from scipy.special import erf, erfinv
from skimage import filters, measure, morphology
import ipywidgets as widgets
from IPython.display import display

try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        pass

pd.set_option('display.max_colwidth', 180)
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['figure.figsize'] = (7, 5)


def read_image_2d(path):
    arr = np.asarray(tifffile.imread(str(path)))
    if arr.ndim == 2:
        img = arr
    elif arr.ndim == 3 and arr.shape[-1] in (3, 4):
        img = arr[..., :3].mean(axis=-1)
    elif arr.ndim == 3:
        img = arr.sum(axis=0)
    else:
        raise ValueError(f'Unsupported image shape {arr.shape} for {path}')
    return img.astype(np.float32, copy=False)


def robust_limits(img, percentiles=(0.5, 99.7)):
    vals = np.asarray(img)[np.isfinite(img)]
    if vals.size == 0:
        return 0.0, 1.0
    lo, hi = np.percentile(vals, percentiles)
    if not np.isfinite(hi - lo) or hi <= lo:
        lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
    if hi <= lo:
        hi = lo + 1.0
    return float(lo), float(hi)


def block_mean_downsample(img, factor):
    factor = int(max(1, factor))
    if factor == 1:
        return np.asarray(img)
    h, w = img.shape
    hh = h // factor * factor
    ww = w // factor * factor
    if hh == 0 or ww == 0:
        return np.asarray(img)
    return img[:hh, :ww].reshape(hh // factor, factor, ww // factor, factor).mean(axis=(1, 3))


def clip_crop_bounds(cx, cy, half_width, shape):
    h, w = shape
    x0 = max(0, int(round(cx - half_width)))
    x1 = min(w, int(round(cx + half_width + 1)))
    y0 = max(0, int(round(cy - half_width)))
    y1 = min(h, int(round(cy + half_width + 1)))
    return x0, x1, y0, y1


def fit_circle_lstsq(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 3:
        raise ValueError('Need at least three points to fit a circle')
    A = np.column_stack([x, y, np.ones_like(x)])
    b = x * x + y * y
    a, bcoef, c = np.linalg.lstsq(A, b, rcond=None)[0]
    cx = a / 2.0
    cy = bcoef / 2.0
    radius_sq = c + cx * cx + cy * cy
    if radius_sq <= 0:
        raise ValueError('Circle fit produced a non-positive radius')
    return float(cx), float(cy), float(np.sqrt(radius_sq))


def nearest_true_pixel(mask, x, y, max_radius):
    yy, xx = np.nonzero(mask)
    if xx.size == 0:
        return None
    dist2 = (xx - x) ** 2 + (yy - y) ** 2
    i = int(np.argmin(dist2))
    if dist2[i] <= max_radius ** 2:
        return int(xx[i]), int(yy[i])
    return None


def component_from_seed(img, seed_xy, search_radius_px=260, polarity='bright', smooth_sigma_px=1.2,
                        min_area_px=80, seed_snap_radius_px=25):
    sx, sy = map(float, seed_xy)
    x0, x1, y0, y1 = clip_crop_bounds(sx, sy, search_radius_px, img.shape)
    crop = np.asarray(img[y0:y1, x0:x1], dtype=np.float32)
    work = ndi.gaussian_filter(crop, smooth_sigma_px) if smooth_sigma_px > 0 else crop
    finite = np.isfinite(work)
    if not finite.any():
        raise ValueError('Local crop contains no finite pixels')

    vals = work[finite]
    lo, hi = np.percentile(vals, [1, 99.5])
    if hi <= lo:
        hi = float(np.max(vals))
        lo = float(np.min(vals))
    scaled = np.clip((work - lo) / max(hi - lo, 1e-6), 0, 1)
    try:
        threshold = filters.threshold_otsu(scaled[finite])
    except Exception:
        threshold = 0.5

    if polarity.lower().startswith('bright'):
        mask = scaled > threshold
    elif polarity.lower().startswith('dark'):
        mask = scaled < threshold
    else:
        raise ValueError("FEATURE_POLARITY must be 'bright' or 'dark'")

    mask = morphology.remove_small_objects(mask.astype(bool), min_size=max(4, int(min_area_px // 4)))
    mask = ndi.binary_closing(mask, structure=np.ones((3, 3), dtype=bool))
    mask = ndi.binary_fill_holes(mask)

    lx = int(round(sx - x0))
    ly = int(round(sy - y0))
    lx = int(np.clip(lx, 0, mask.shape[1] - 1))
    ly = int(np.clip(ly, 0, mask.shape[0] - 1))

    if not mask[ly, lx]:
        snapped = nearest_true_pixel(mask, lx, ly, seed_snap_radius_px)
        if snapped is None:
            raise ValueError('Clicked seed is not on the thresholded circle and no nearby circle pixel was found')
        lx, ly = snapped

    labels = measure.label(mask)
    lab = labels[ly, lx]
    if lab == 0:
        raise ValueError('No connected component found at the clicked seed')
    comp = labels == lab
    area = int(comp.sum())
    if area < min_area_px:
        raise ValueError(f'Connected component is too small ({area} px)')

    boundary = comp ^ ndi.binary_erosion(comp)
    by, bx = np.nonzero(boundary)
    if bx.size >= 12:
        try:
            cx_local, cy_local, radius = fit_circle_lstsq(bx, by)
        except Exception:
            cy_local, cx_local = ndi.center_of_mass(comp)
            radius = math.sqrt(area / math.pi)
    else:
        cy_local, cx_local = ndi.center_of_mass(comp)
        radius = math.sqrt(area / math.pi)

    touches_crop_edge = bool(comp[0, :].any() or comp[-1, :].any() or comp[:, 0].any() or comp[:, -1].any())
    return {
        'crop': crop,
        'crop_bounds': (x0, x1, y0, y1),
        'component': comp,
        'component_area_px': area,
        'center_x': float(cx_local + x0),
        'center_y': float(cy_local + y0),
        'radius_px': float(radius),
        'touches_crop_edge': touches_crop_edge,
        'threshold': float(threshold),
    }


def sample_rays(img, cx, cy, radii, angles):
    radii = np.asarray(radii, dtype=float)
    angles = np.asarray(angles, dtype=float)
    xs = cx + radii[:, None] * np.cos(angles)[None, :]
    ys = cy + radii[:, None] * np.sin(angles)[None, :]
    valid = (xs >= 0) & (xs <= img.shape[1] - 1) & (ys >= 0) & (ys <= img.shape[0] - 1)
    vals = ndi.map_coordinates(img, [np.clip(ys, 0, img.shape[0] - 1), np.clip(xs, 0, img.shape[1] - 1)], order=1, mode='nearest')
    vals = vals.astype(np.float32, copy=False)
    vals[~valid] = np.nan
    return vals, valid


def ray_edge_points(img, cx, cy, radius_guess, n_angles=720, radial_step_px=0.25,
                    edge_search_px=18, outside_px=30, smooth_sigma_samples=1.0):
    r_max = max(radius_guess + outside_px + edge_search_px, radius_guess * 1.15)
    radii = np.arange(0, r_max + radial_step_px / 2, radial_step_px)
    angles = np.linspace(0, 2 * np.pi, int(n_angles), endpoint=False)
    vals, valid = sample_rays(img, cx, cy, radii, angles)
    points = []
    edge_radii = []
    for j, theta in enumerate(angles):
        y = vals[:, j]
        ok = np.isfinite(y) & valid[:, j]
        if ok.sum() < 20:
            continue
        yy = y.copy()
        finite_idx = np.where(np.isfinite(yy))[0]
        if finite_idx.size < 20:
            continue
        # Fill sparse invalid samples before smoothing.
        yy[~np.isfinite(yy)] = np.interp(np.where(~np.isfinite(yy))[0], finite_idx, yy[finite_idx])
        if smooth_sigma_samples > 0:
            yy = ndi.gaussian_filter1d(yy, smooth_sigma_samples)
        lo = max(0, radius_guess - edge_search_px)
        hi = min(radii[-1], radius_guess + edge_search_px)
        search = (radii >= lo) & (radii <= hi)
        if search.sum() < 5:
            continue
        grad = np.gradient(yy, radii)
        idxs = np.where(search)[0]
        k = idxs[int(np.argmax(np.abs(grad[idxs])))]
        re = float(radii[k])
        if not np.isfinite(re):
            continue
        points.append((cx + re * np.cos(theta), cy + re * np.sin(theta)))
        edge_radii.append(re)
    return np.asarray(points, dtype=float), np.asarray(edge_radii, dtype=float)


def refine_center_from_edges(img, cx, cy, radius, n_angles=720, radial_step_px=0.25,
                             edge_search_px=18, outside_px=30, max_iter=3):
    warnings_list = []
    for _ in range(int(max_iter)):
        pts, edge_rs = ray_edge_points(img, cx, cy, radius, n_angles=n_angles, radial_step_px=radial_step_px,
                                       edge_search_px=edge_search_px, outside_px=outside_px)
        if pts.shape[0] < max(24, n_angles * 0.10):
            warnings_list.append(f'Only {pts.shape[0]} radial edge points available for center refinement')
            break
        try:
            ncx, ncy, nr = fit_circle_lstsq(pts[:, 0], pts[:, 1])
        except Exception as exc:
            warnings_list.append(f'Circle fit from radial edge points failed: {exc}')
            break
        shift = math.hypot(ncx - cx, ncy - cy)
        cx, cy, radius = ncx, ncy, nr
        if shift < 0.05:
            break
    return float(cx), float(cy), float(radius), warnings_list


def radial_profile(img, cx, cy, r_max, n_angles=720, radial_step_px=0.25):
    radii = np.arange(0, r_max + radial_step_px / 2, radial_step_px)
    angles = np.linspace(0, 2 * np.pi, int(n_angles), endpoint=False)
    vals, valid = sample_rays(img, cx, cy, radii, angles)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        prof_median = np.nanmedian(vals, axis=1)
        prof_mean = np.nanmean(vals, axis=1)
    valid_fraction = np.isfinite(vals).mean(axis=1)
    return pd.DataFrame({'radius_px': radii, 'median_intensity': prof_median, 'mean_intensity': prof_mean, 'valid_fraction': valid_fraction})


def esf_model(x, inside, outside, x0, sigma):
    sigma = np.maximum(sigma, 1e-6)
    return outside + (inside - outside) * 0.5 * (1.0 - erf((x - x0) / (np.sqrt(2.0) * sigma)))


def fit_esf(profile_df, radius_px, edge_window_in_px=25, edge_window_out_px=30):
    signed = profile_df['radius_px'].to_numpy() - radius_px
    y = profile_df['median_intensity'].to_numpy(dtype=float)
    ok = np.isfinite(signed) & np.isfinite(y) & (signed >= -edge_window_in_px) & (signed <= edge_window_out_px)
    x = signed[ok]
    yy = y[ok]
    if x.size < 16:
        raise ValueError('Not enough finite ESF samples in the requested edge window')

    inside_band = yy[x < -0.35 * edge_window_in_px]
    outside_band = yy[x > 0.35 * edge_window_out_px]
    inside0 = float(np.nanmedian(inside_band)) if inside_band.size else float(np.nanpercentile(yy, 80))
    outside0 = float(np.nanmedian(outside_band)) if outside_band.size else float(np.nanpercentile(yy, 20))
    sigma0 = max(1.0, min(edge_window_in_px, edge_window_out_px) / 6.0)
    p0 = [inside0, outside0, 0.0, sigma0]

    y_min = float(np.nanmin(yy))
    y_max = float(np.nanmax(yy))
    pad = max(1e-6, 2.0 * (y_max - y_min))
    bounds = ([y_min - pad, y_min - pad, -min(edge_window_in_px, 8), 0.15],
              [y_max + pad, y_max + pad, min(edge_window_out_px, 8), max(edge_window_in_px, edge_window_out_px)])

    popt, pcov = curve_fit(esf_model, x, yy, p0=p0, bounds=bounds, maxfev=20000)
    fit_y = esf_model(x, *popt)
    resid = yy - fit_y
    rmse = float(np.sqrt(np.nanmean(resid ** 2)))
    inside, outside, x0, sigma = [float(v) for v in popt]
    mtf10 = float(np.sqrt(np.log(10.0)) / (np.sqrt(2.0) * np.pi * sigma))
    resolution_10pct = float(0.5 / mtf10)
    fwhm = float(2.0 * np.sqrt(2.0 * np.log(2.0)) * sigma)
    edge_10_90 = float(2.0 * np.sqrt(2.0) * erfinv(0.8) * sigma)
    return {
        'x_px': x,
        'y': yy,
        'fit_y': fit_y,
        'inside_intensity': inside,
        'outside_intensity': outside,
        'edge_offset_px': x0,
        'sigma_lsf_px': sigma,
        'fwhm_lsf_px': fwhm,
        'edge_10_90_px': edge_10_90,
        'mtf10_lp_per_px': mtf10,
        'resolution_mtf10_px': resolution_10pct,
        'fit_rmse': rmse,
    }


def analyze_circle(img, seed_xy, circle_id=0):
    notes = []
    comp = component_from_seed(
        img, seed_xy,
        search_radius_px=SEARCH_RADIUS_PX,
        polarity=FEATURE_POLARITY,
        smooth_sigma_px=SMOOTH_SIGMA_PX,
        min_area_px=MIN_COMPONENT_AREA_PX,
        seed_snap_radius_px=SEED_SNAP_RADIUS_PX,
    )
    if comp['touches_crop_edge']:
        notes.append('component touches local crop edge; consider increasing SEARCH_RADIUS_PX')

    cx, cy, radius = comp['center_x'], comp['center_y'], comp['radius_px']
    cx, cy, radius, refine_notes = refine_center_from_edges(
        img, cx, cy, radius,
        n_angles=N_ANGLES,
        radial_step_px=RADIAL_STEP_PX,
        edge_search_px=EDGE_POINT_SEARCH_PX,
        outside_px=EDGE_WINDOW_OUT_PX,
        max_iter=MAX_CENTER_REFINE_ITER,
    )
    notes.extend(refine_notes)

    r_max = radius + EDGE_WINDOW_OUT_PX
    prof = radial_profile(img, cx, cy, r_max, n_angles=N_ANGLES, radial_step_px=RADIAL_STEP_PX)
    fit = fit_esf(prof, radius, edge_window_in_px=EDGE_WINDOW_IN_PX, edge_window_out_px=EDGE_WINDOW_OUT_PX)

    result = {
        'circle_id': int(circle_id),
        'seed_x': float(seed_xy[0]),
        'seed_y': float(seed_xy[1]),
        'center_x': cx,
        'center_y': cy,
        'radius_px': radius,
        'component_area_px': comp['component_area_px'],
        'sigma_lsf_px': fit['sigma_lsf_px'],
        'fwhm_lsf_px': fit['fwhm_lsf_px'],
        'edge_10_90_px': fit['edge_10_90_px'],
        'mtf10_lp_per_px': fit['mtf10_lp_per_px'],
        'resolution_mtf10_px': fit['resolution_mtf10_px'],
        'edge_offset_px': fit['edge_offset_px'],
        'fit_rmse': fit['fit_rmse'],
        'status': 'ok',
        'notes': '; '.join(notes),
    }
    if PIXEL_SIZE_UM is not None:
        px = float(PIXEL_SIZE_UM)
        result.update({
            'sigma_lsf_um': result['sigma_lsf_px'] * px,
            'fwhm_lsf_um': result['fwhm_lsf_px'] * px,
            'edge_10_90_um': result['edge_10_90_px'] * px,
            'mtf10_lp_per_mm': result['mtf10_lp_per_px'] * 1000.0 / px,
            'resolution_mtf10_um': result['resolution_mtf10_px'] * px,
        })
    return result, prof, fit


def analyze_points(img, points_xy):
    rows = []
    profiles = {}
    fits = {}
    for i, pt in enumerate(points_xy, start=1):
        try:
            row, prof, fit = analyze_circle(img, pt, circle_id=i)
            profiles[i] = prof
            fits[i] = fit
        except Exception as exc:
            row = {
                'circle_id': i,
                'seed_x': float(pt[0]),
                'seed_y': float(pt[1]),
                'center_x': np.nan,
                'center_y': np.nan,
                'radius_px': np.nan,
                'sigma_lsf_px': np.nan,
                'fwhm_lsf_px': np.nan,
                'edge_10_90_px': np.nan,
                'mtf10_lp_per_px': np.nan,
                'resolution_mtf10_px': np.nan,
                'edge_offset_px': np.nan,
                'fit_rmse': np.nan,
                'status': 'failed',
                'notes': str(exc),
            }
        rows.append(row)
    return pd.DataFrame(rows), profiles, fits


def plot_circle_result(img, results_df, profiles, fits, circle_id, crop_pad_px=45):
    row = results_df.loc[results_df['circle_id'] == circle_id].iloc[0]
    if row['status'] != 'ok':
        print(row[['circle_id', 'status', 'notes']])
        return
    cx, cy, radius = float(row.center_x), float(row.center_y), float(row.radius_px)
    x0, x1, y0, y1 = clip_crop_bounds(cx, cy, radius + crop_pad_px, img.shape)
    crop = img[y0:y1, x0:x1]
    vmin, vmax = robust_limits(crop, DISPLAY_PERCENTILES)

    prof = profiles[int(circle_id)]
    fit = fits[int(circle_id)]
    signed = prof['radius_px'].to_numpy() - radius

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
    ax = axes[0]
    ax.imshow(crop, vmin=vmin, vmax=vmax, extent=(x0, x1, y1, y0), origin='upper')
    circle = plt.Circle((cx, cy), radius, fill=False, color='tab:red', lw=1.5)
    ax.add_patch(circle)
    ax.plot([row.seed_x], [row.seed_y], 'x', color='tab:blue', ms=7, label='seed')
    ax.plot([cx], [cy], '+', color='yellow', ms=9, mew=1.5, label='center')
    ax.set_title(f'Circle {circle_id}')
    ax.set_xlim(x0, x1)
    ax.set_ylim(y1, y0)
    ax.legend(loc='lower right')

    ax = axes[1]
    ax.plot(signed, prof['median_intensity'], '.', ms=3, alpha=0.65, label='radial median ESF')
    order = np.argsort(fit['x_px'])
    ax.plot(np.asarray(fit['x_px'])[order], np.asarray(fit['fit_y'])[order], 'r-', lw=2, label='erf fit')
    ax.axvline(0, color='k', lw=1, ls='--', label='edge radius')
    ax.set_xlim(-EDGE_WINDOW_IN_PX, EDGE_WINDOW_OUT_PX)
    ax.set_xlabel('Distance from fitted circle edge [px]')
    ax.set_ylabel('Intensity')
    ax.set_title('Radial ESF')
    ax.legend()

    ax = axes[2]
    x = np.linspace(-EDGE_WINDOW_IN_PX, EDGE_WINDOW_OUT_PX, 600)
    inside = fit['inside_intensity']
    outside = fit['outside_intensity']
    x0fit = fit['edge_offset_px']
    sigma = fit['sigma_lsf_px']
    lsf = np.abs((inside - outside) / (np.sqrt(2 * np.pi) * sigma) * np.exp(-0.5 * ((x - x0fit) / sigma) ** 2))
    if np.nanmax(lsf) > 0:
        lsf = lsf / np.nanmax(lsf)
    ax.plot(x, lsf, 'm-', lw=2)
    ax.axvline(x0fit, color='k', lw=1, ls='--')
    ax.set_xlabel('Distance from edge [px]')
    ax.set_ylabel('Normalized |LSF|')
    ax.set_title(f"sigma={row.sigma_lsf_px:.2f} px, FWHM={row.fwhm_lsf_px:.2f} px")
    ax.set_ylim(0, 1.08)
    fig.tight_layout()
    return fig


## 3. Load and display the image

In [ ]:
img = read_image_2d(IMAGE_PATH)
vmin, vmax = robust_limits(img, DISPLAY_PERCENTILES)
print(f'Loaded {IMAGE_PATH}')
print(f'shape={img.shape}, dtype={img.dtype}, display vmin={vmin:.4g}, vmax={vmax:.4g}')

fig, ax = plt.subplots(figsize=(7, 7))
preview = block_mean_downsample(img, DISPLAY_DOWNSAMPLE)
ax.imshow(preview, vmin=vmin, vmax=vmax, extent=(0, img.shape[1], img.shape[0], 0), origin='upper')
ax.set_title('Preview')
ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')
fig.tight_layout()

## 4. Click one point inside each circle

Run the cell below, then left-click once inside every circle you want to measure. Use **Undo** if you mis-click. The stored coordinates are full-resolution image coordinates even when the displayed image is downsampled.

In [ ]:

class CirclePointPicker:
    def __init__(self, img, downsample=2, vmin=None, vmax=None):
        self.img = img
        self.points = []
        self.finished = False
        self.preview = block_mean_downsample(img, downsample)
        self.fig, self.ax = plt.subplots(figsize=(8, 8))
        self.ax.imshow(self.preview, vmin=vmin, vmax=vmax, extent=(0, img.shape[1], img.shape[0], 0), origin='upper')
        self.ax.set_title('Left-click inside each circle')
        self.ax.set_xlabel('x [px]')
        self.ax.set_ylabel('y [px]')
        self.scatter = self.ax.scatter([], [], s=36, facecolors='none', edgecolors='tab:red', linewidths=1.5)
        self.labels = []
        self.status = widgets.HTML()
        self.undo_button = widgets.Button(description='Undo', icon='undo')
        self.clear_button = widgets.Button(description='Clear', icon='trash')
        self.finish_button = widgets.Button(description='Finish', icon='check')
        self.undo_button.on_click(self.undo)
        self.clear_button.on_click(self.clear)
        self.finish_button.on_click(self.finish)
        display(widgets.HBox([self.undo_button, self.clear_button, self.finish_button]), self.status)
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        self.update_plot()

    def on_click(self, event):
        if self.finished or event.inaxes is not self.ax or event.button != 1:
            return
        if event.xdata is None or event.ydata is None:
            return
        self.points.append((float(event.xdata), float(event.ydata)))
        self.update_plot()

    def update_plot(self):
        xy = np.asarray(self.points, dtype=float) if self.points else np.empty((0, 2))
        self.scatter.set_offsets(xy)
        for text in self.labels:
            text.remove()
        self.labels = []
        for i, (x, y) in enumerate(self.points, start=1):
            self.labels.append(self.ax.text(x + 8, y + 8, str(i), color='yellow', fontsize=9, weight='bold'))
        self.status.value = f'<b>{len(self.points)}</b> point(s) selected'
        self.fig.canvas.draw_idle()

    def undo(self, _=None):
        if self.points and not self.finished:
            self.points.pop()
            self.update_plot()

    def clear(self, _=None):
        if not self.finished:
            self.points = []
            self.update_plot()

    def finish(self, _=None):
        self.finished = True
        self.fig.canvas.mpl_disconnect(self.cid)
        self.status.value = f'<b>{len(self.points)}</b> point(s) selected. Run the next cell to analyze them.'

    def to_dataframe(self):
        return pd.DataFrame(self.points, columns=['seed_x', 'seed_y'])

picker = CirclePointPicker(img, downsample=DISPLAY_DOWNSAMPLE, vmin=vmin, vmax=vmax)


## 5. Review or manually edit selected points

If the interactive picker does not work in your Jupyter frontend, fill `MANUAL_POINTS_XY` with `(x, y)` pairs and set `USE_MANUAL_POINTS = True`.

In [ ]:
USE_MANUAL_POINTS = False
MANUAL_POINTS_XY = [
    # (x, y),
]

if USE_MANUAL_POINTS:
    points_df = pd.DataFrame(MANUAL_POINTS_XY, columns=['seed_x', 'seed_y'])
else:
    points_df = picker.to_dataframe()

points_df.to_csv(OUTPUT_DIR / 'circle_seed_points.csv', index=False)
print(f'{len(points_df)} selected point(s). Saved seeds to {OUTPUT_DIR / "circle_seed_points.csv"}')
points_df

## 6. Analyze all selected circles

This cell finds each center, extracts each radial profile, fits the ESF, and saves the summary table. Failed rows usually mean the seed was outside the thresholded circle, the crop was too small, or the circle was too close to the image border.

In [ ]:
points_xy = list(points_df[['seed_x', 'seed_y']].itertuples(index=False, name=None))
results_df, profiles, fits = analyze_points(img, points_xy)

results_path = OUTPUT_DIR / 'circle_resolution_results.csv'
results_df.to_csv(results_path, index=False)
print(f'Saved results to {results_path}')

ok = results_df['status'].eq('ok')
print(f'{ok.sum()} successful / {len(results_df)} selected circles')
if ok.any():
    summary_cols = ['sigma_lsf_px', 'fwhm_lsf_px', 'edge_10_90_px', 'resolution_mtf10_px']
    display(results_df.loc[ok, summary_cols].describe().T)
results_df

## 7. Inspect individual fits

Use the dropdown to inspect the refined center/radius, radial ESF, and fitted line-spread function for each circle. Bad-looking fits should be removed from the seed list or flagged before taking averages.

In [ ]:
valid_ids = results_df.loc[results_df['status'].eq('ok'), 'circle_id'].astype(int).tolist()
if not valid_ids:
    print('No successful circle fits to inspect.')
else:
    circle_dropdown = widgets.Dropdown(options=valid_ids, value=valid_ids[0], description='Circle')
    crop_pad_slider = widgets.IntSlider(value=45, min=15, max=160, step=5, description='Crop pad')

    def show_fit(circle_id, crop_pad_px):
        plt.close('all')
        plot_circle_result(img, results_df, profiles, fits, int(circle_id), crop_pad_px=int(crop_pad_px))
        plt.show()

    display(widgets.interactive_output(show_fit, {'circle_id': circle_dropdown, 'crop_pad_px': crop_pad_slider}))
    display(widgets.HBox([circle_dropdown, crop_pad_slider]))

## 8. Export radial profiles for accepted circles

Set `ACCEPTED_CIRCLE_IDS` if you want to export only a curated subset after inspection. Leave it as `None` to export all successful fits.

In [ ]:
ACCEPTED_CIRCLE_IDS = None  # example: [1, 2, 5, 8]

if ACCEPTED_CIRCLE_IDS is None:
    export_ids = results_df.loc[results_df['status'].eq('ok'), 'circle_id'].astype(int).tolist()
else:
    export_ids = [int(x) for x in ACCEPTED_CIRCLE_IDS]

profile_rows = []
for cid in export_ids:
    if cid not in profiles:
        continue
    tmp = profiles[cid].copy()
    radius = float(results_df.loc[results_df['circle_id'].eq(cid), 'radius_px'].iloc[0])
    tmp.insert(0, 'circle_id', cid)
    tmp['distance_from_edge_px'] = tmp['radius_px'] - radius
    profile_rows.append(tmp)

if profile_rows:
    profiles_df = pd.concat(profile_rows, ignore_index=True)
    profiles_path = OUTPUT_DIR / 'circle_radial_profiles.csv'
    profiles_df.to_csv(profiles_path, index=False)
    print(f'Saved radial profiles to {profiles_path}')
    display(profiles_df.head())
else:
    print('No profiles exported.')

## Notes on the reported resolution numbers

`sigma_lsf_px` is the Gaussian sigma of the line-spread function implied by the fitted radial edge-spread function. `fwhm_lsf_px` is `2.355 * sigma`. `edge_10_90_px` is the 10-90 percent edge width. `mtf10_lp_per_px` is the spatial frequency where a Gaussian MTF reaches 10 percent contrast, and `resolution_mtf10_px` is the corresponding half-period length, `0.5 / mtf10`. If `PIXEL_SIZE_UM` is set, the notebook also reports micrometers and lp/mm.